<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 9 · Reinforcement Learning for Trading Decisions

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook unpacks the tabular Q-learning example from Chapter 9.

- Prepare the state space based on z-scored returns and positions.
- Train a small Q-learning agent on SPY data with reduced episode count.
- Compare the learned policy’s equity curve with buy-and-hold.


In [ ]:
import sys
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch09_rl_baseline as ch09  # RL helpers and backtest utilities
import ch07_baseline_strategies as ch07  # benchmark backtest helpers


### 1. Prepare RL Data

We construct prices, returns, and discretised z-scores exactly as in the standalone script, but we will train for far fewer episodes to keep notebooks responsive.

In [ ]:
prices, rets, z = ch09.prepare_rl_data(symbol="EURUSD", z_window=60)
z_bins = ch09.discretise_z(z)

rets_train = rets.iloc[:500]
z_bins_train = z_bins.iloc[:500]

prices_test = prices.loc[rets.index[500:]]
rets_test = rets.iloc[500:]

prices.head(), rets_train.shape, rets_test.shape


### 2. Train a Small Q-Learning Agent

For illustration we reduce the number of episodes; the code structure mirrors the full experiment and highlights the separation between environment data and learning algorithm.

In [ ]:
Q, episode_means = ch09.train_q_learning(
    rets=rets_train,
    z_bins=z_bins_train,
    episodes=250,
    alpha=0.05,
    gamma=0.98,
    epsilon_start=1.0,
    epsilon_end=0.05,
    log_progress=False,
)

len(episode_means), Q.shape


### 3. Greedy Policy vs. Buy-and-Hold

We extract the greedy policy implied by the learned Q-table and compare its equity curve with a classic buy-and-hold benchmark on the test window.

In [ ]:
pos_rl = ch09.build_greedy_positions(
    rets=rets_test,
    z_bins=z_bins.loc[rets_test.index],
    Q=Q,
)

pos_bh = ch07.build_positions_buy_and_hold(prices_test)

net_bh = ch07.backtest_strategy(prices_test, pos_bh)
net_rl = ch07.backtest_strategy(prices_test, pos_rl)

wealth_bh = (1.0 + net_bh).cumprod()
wealth_rl = (1.0 + net_rl).cumprod()

fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot(wealth_bh.index, wealth_bh.values, label="buy-and-hold")
ax.plot(wealth_rl.index, wealth_rl.values, label="RL policy")
ax.set_ylabel("wealth (normalised)")
ax.set_xlabel("date")
ax.set_title("EURUSD: RL policy vs buy-and-hold (toy example)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- Reinforcement learning reframes trading as a sequence of state–action–reward decisions.
- The Chapter 9 script keeps the environment intentionally simple so that the learning logic is easy to inspect.
- Later chapters can reuse this structure with richer state representations, reward definitions, and function approximators.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">